# D4Explainer — Colab Setup
**Runtime:** GPU **A100 (40 GB)** required for paper-faithful runs — set via *Runtime > Change runtime type*.

T4 (16 GB) OOMs on every dataset under the paper-faithful loop because the Powerful net's `[bsz × sigma_length, N, N, hidden × num_layers]` activation tensor for one `backward()` exceeds 14 GB even at paper bsz (empirically confirmed on BA_shapes, bsz=4, padded N≈500, ~15 GB peak). Switching to A100 is a compute fix that preserves the paper's training procedure exactly; falling back to `--memory_efficient` on T4 is also possible but introduces a documented CF-loss approximation.

In [ ]:
# Cell 1: Upload and unzip code
from google.colab import files
import zipfile, os

print('Select D4Explainer.zip when the picker opens...')
uploaded = files.upload()  # select D4Explainer.zip

with zipfile.ZipFile('D4Explainer.zip', 'r') as z:
    z.extractall('/content/')

os.chdir('/content/D4Explainer')
print('CWD:', os.getcwd())

In [ ]:
# Cell 2: Upload and unzip datasets
# Upload one zip per dataset — repeat for each: MUTAG, BA3, BA_shapes, etc.
from google.colab import files
import zipfile, os

os.makedirs('/content/D4Explainer/data', exist_ok=True)

print('Select dataset zip(s) when the picker opens...')
uploaded = files.upload()  # can select multiple files

for fname in uploaded:
    if fname.endswith('.zip'):
        with zipfile.ZipFile(fname, 'r') as z:
            z.extractall('/content/D4Explainer/data/')
        print(f'Extracted {fname}')

print('\ndata/ contents:', os.listdir('/content/D4Explainer/data'))

In [ ]:
# Cell 3: Verify GPU
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    print('Device:', name)
    if 'A100' not in name:
        print(f'WARNING: paper-faithful loop expects A100 (40 GB). Current device "{name}" '
              f'will likely OOM on the BA_shapes smoke test (~15 GB peak activations).')
else:
    print('No GPU — check Runtime > Change runtime type > A100 GPU')

In [ ]:
# Cell 3b: Prevent TensorFlow GPU pre-allocation and verify clean GPU state
# TF is installed in Colab alongside PyTorch and grabs all GPU memory by default.
# Must run this before any training — if GPU memory looks wrong, restart the runtime first.
import os
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'

import torch
allocated = torch.cuda.memory_allocated() / 1024**3
reserved  = torch.cuda.memory_reserved()  / 1024**3
total     = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Total VRAM : {total:.1f} GiB')
print(f'Allocated  : {allocated:.2f} GiB  (should be ~0 on a fresh runtime)')
print(f'Reserved   : {reserved:.2f} GiB')
if allocated > 1.0:
    print('WARNING: >1 GiB already allocated — restart the runtime to clear leftover memory.')

In [ ]:
# Cell 4: Install dependencies
# Note: torch-scatter/sparse/cluster/spline-conv are pre-installed on modern Colab runtimes.
# torch-geometric version pin dropped — 2.4.0 has no wheel for Python 3.12+.
# pandas/networkx version pins dropped — pinned versions predate Python 3.12.
# rdkit-pypi renamed to rdkit.
import torch, subprocess

TORCH_VERSION = torch.__version__.split('+')[0]
result = subprocess.run(['nvcc', '--version'], capture_output=True, text=True)
cuda_str = result.stdout.split('release ')[-1].split(',')[0].strip().replace('.', '')
CUDA_TAG = f'cu{cuda_str}'
print(f'Torch: {TORCH_VERSION}, CUDA: {CUDA_TAG}')

# PyG sparse extensions (likely pre-installed; no-op if already present)
!pip install -q torch-scatter torch-sparse torch-cluster torch-spline-conv \
    -f https://data.pyg.org/whl/torch-{TORCH_VERSION}+{CUDA_TAG}.html

!pip install -q torch-geometric
!pip install -q networkx pandas rdkit pyemd

In [ ]:
# Cell 5: Smoke-test imports
import torch_geometric
import networkx, pandas, numpy
print('PyTorch:   ', torch.__version__)
print('PyG:       ', torch_geometric.__version__)
print('NetworkX:  ', networkx.__version__)
print('pandas:    ', pandas.__version__)
print('numpy:     ', numpy.__version__)
print('All imports OK')

In [ ]:
# Cell 6: Verify trained GNN checkpoints exist for every dataset.
import os
ckpts = [
    'param/gnns/BA_shapes_gcn.pt',
    'param/gnns/Tree_Cycle_gcn.pt',
    'param/gnns/Tree_Grids_gcn.pt',
    'param/gnns/cornell_gcn.pt',
    'param/gnns/NCI1_gcn.pt',
    'param/gnns/bbbp_gcn.pt',
    'param/gnns/ba3_gcn.pt',
    'param/gnns/mutag_gcn.pt',
]
missing = [c for c in ckpts if not os.path.exists(c)]
assert not missing, f'Missing checkpoints: {missing} — include them in the zip or train via gnns/'
print(f'All {len(ckpts)} GNN checkpoints found.')


In [ ]:
# Cell 6b: Mount Google Drive and symlink results/ -> Drive so run outputs survive runtime resets.
# All training cells write to results/{dataset}/{run_name}/{config.json,metrics.jsonl,best_model.pth};
# the symlink redirects those writes to Drive transparently.
#
# Self-heals if local results/ has contents (e.g. partial outputs from a prior in-session run
# that wrote before Cell 6b ran, or a zip that did not fully exclude results/):
# files are merged into Drive with skip-if-exists semantics (Drive-side files always win),
# then the local dir is wiped and replaced by the symlink.
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
DRIVE_RESULTS = '/content/drive/MyDrive/D4Explainer_results'
os.makedirs(DRIVE_RESULTS, exist_ok=True)

local = '/content/D4Explainer/results'


def merge_into_drive(src, dst):
    """Copy each file from src tree into dst tree only if it does not already exist in dst.
    Drive-side wins on any name collision so committed runs are never overwritten by stale
    local copies (e.g. partial outputs from an OOMed run)."""
    copied = skipped = 0
    for root, _, files in os.walk(src):
        rel = os.path.relpath(root, src)
        dst_root = os.path.join(dst, rel) if rel != '.' else dst
        os.makedirs(dst_root, exist_ok=True)
        for f in files:
            sp = os.path.join(root, f)
            dp = os.path.join(dst_root, f)
            if os.path.exists(dp):
                skipped += 1
            else:
                shutil.copy2(sp, dp)
                copied += 1
    return copied, skipped


if os.path.islink(local):
    os.unlink(local)
    os.symlink(DRIVE_RESULTS, local)
    print(f'Re-symlinked {local} -> {DRIVE_RESULTS}')
elif os.path.isdir(local):
    contents = os.listdir(local)
    if len(contents) == 0:
        os.rmdir(local)
        os.symlink(DRIVE_RESULTS, local)
        print(f'Removed empty real dir at {local} and symlinked -> {DRIVE_RESULTS}')
    else:
        print(f'Found {len(contents)} item(s) in local {local}: {contents}')
        print(f'Merging into {DRIVE_RESULTS} (skip-if-exists; Drive-side files preserved)...')
        copied, skipped = merge_into_drive(local, DRIVE_RESULTS)
        print(f'  copied {copied} new file(s); skipped {skipped} (already in Drive)')
        shutil.rmtree(local)
        os.symlink(DRIVE_RESULTS, local)
        print(f'Wiped local dir and symlinked -> {DRIVE_RESULTS}')
elif not os.path.exists(local):
    os.symlink(DRIVE_RESULTS, local)
    print(f'Symlinked {local} -> {DRIVE_RESULTS}')

# Verify the write actually lands in Drive (not ephemeral local disk).
test_path = os.path.join(local, '.write_test')
drive_test = os.path.join(DRIVE_RESULTS, '.write_test')
with open(test_path, 'w') as f:
    f.write('ok')
assert os.path.exists(drive_test), (
    f'WRITE LANDED ON LOCAL DISK, NOT DRIVE: {test_path} exists but {drive_test} does not. '
    f'Symlink is not redirecting writes - investigate before training.'
)
os.remove(test_path)
print(f'Drive write test verified end-to-end. Run outputs will persist to: {DRIVE_RESULTS}')


## Deviations from the original D4Explainer paper

The **default** training loop is now **paper-faithful** — single batched-sigma forward, one `loss.backward()`, CF loss averaged exactly over all 10 sigmas (matches commit `2cead28`). All memory-workaround changes below are now opt-in behind `--memory_efficient`.

### Hardware requirement (empirical, 2026-04-26)

The paper-faithful loop requires **A100 (40 GB)**. T4 (16 GB) OOMs on every dataset tried (confirmed on BA_shapes at paper bsz=4, padded N≈500: peak `[bsz × sigma_length, N, N, hidden × num_layers] = [40, 498, 498, 384] ≈ 15 GB`). This is a **compute** issue, not a methodological one — preserving the paper's training procedure on T4 would require either `--memory_efficient` (CF-loss approximation, see below) or aggressive `--max_graph_size` filtering (data change). A100 is the clean fix.

### Opt-in memory-mitigation stack (`--memory_efficient` path only)

| Change | Flag | Fidelity | Notes |
|---|---|---|---|
| **Per-sigma BCE loss** | `--memory_efficient` | **Exact** | Original: one forward over all 10 sigmas, one backward. Memory-efficient: 10 forward+backward, each BCE scaled by `1/sigma_length`. Sum of gradients is algebraically identical. `total_sigmas` kwarg on `loss_func_bce` preserves the `1/|Σ|` weight. |
| **CF loss: single-sigma approximation** | `--memory_efficient` | **Approximation** | Original: CF loss uses the mean of scores across all 10 sigmas as soft edge weights into the frozen GNN. Memory-efficient: CF loss uses only the last sigma's score. Same expected direction but higher-variance gradient estimate. May lower reported fidelity slightly vs paper numbers. |
| **Gradient checkpointing on Powerful** | `--gradient_checkpointing` (only applied under `--memory_efficient`) | **Exact (math); ~2× compute** | Recomputes the full `Powerful` forward during backward. No numerical change, just compute/memory tradeoff. |
| **Mixed-precision (fp16) training** | `--use_amp` (only applied under `--memory_efficient`) | **Approximation (numerical)** | ~2× memory reduction on activations. Small numerical drift vs fp32 baseline. |

### Data-path changes (work in either mode)

| Change | Flag | Fidelity | Notes |
|---|---|---|---|
| **Reduced batch size** | `--train_batchsize` | **Exact (loss); different SGD noise** | Changing bsz changes SGD variance and effective LR. Paper Table 7 bsz per-dataset is documented in README. |
| **Size-bucketed batch sampler** | `--size_bucketed` | **Exact (loss); different SGD sampling** | Groups graphs by node count so each batch pads to a tight N. Changes batch composition but loss per sample is unchanged. Useful for Mutagenicity if you bump bsz above the paper's 2. |
| **Max-graph-size filter** | `--max_graph_size` | **Data change** | Drops graphs above a node-count threshold from **all splits (train/val/test)** so train and eval distributions match. Hardware-driven cap; required for BA_shapes on A100 (cap=450 drops upper ~30%). Document the kept-fraction per split when reporting results. |
| **--debug_shapes** | `--debug_shapes` | N/A (diagnostic only) | Prints padded N and CUDA memory for the first 5 batches. No training effect. |

### Dataset naming caveat
The repo's `--dataset mutag` flag loads **Mutagenicity** (4337 graphs, max ~417 nodes), *not* classic TU MUTAG (188 graphs, max 28). The D4Explainer authors preserved this imprecise label; both the code and the paper tables refer to "MUTAG" but train on Mutagenicity.

### Recommended baseline-vs-extension protocol
Default (paper-faithful, A100):
- No `--memory_efficient`, no `--use_amp`, no `--gradient_checkpointing`
- Paper Table 7 hyperparameters per-dataset (bsz + alpha_cf; see README)

Then run the natural-gradient extension with the **same** flags + `--natural_gradient`. Single-flag toggle keeps the comparison clean.

If A100 is unavailable and you must use T4, fall back to `--memory_efficient` — matched on both baseline and extension. Document this as a compute fallback that introduces the CF-loss single-sigma approximation noted above.

---

### Extension (NOT paper — novel contribution)

| Change | Flag | Fidelity | Notes |
|---|---|---|---|
| **Natural-gradient hook on edge probabilities** | `--natural_gradient` | **Extension** | Registers a backward hook on each `score_batch` tensor that rescales `dL/d(score)` by the diagonal Fisher-Rao inverse metric `I(θ)^{-1} = θ(1-θ)`. Per CONTEXT.md §6 Alg 2: each score tensor is a point on the Bernoulli-product manifold `M_G`, so the natural-gradient correction is element-wise — O(|E|), not O(|E|²). Applied to all forward passes in both training paths. No change to loss, architecture, or data. |
| **Natural-gradient boundary clamp** | `--nat_grad_eps` (default 1e-6) | **Extension hyperparameter** | θ is clamped to `[eps, 1-eps]` inside the hook to avoid vanishing/exploding gradient at the manifold boundary. Rule-of-thumb default; tune if saturation is observed. |

**Experimental protocol:**
- Baseline run: paper-faithful default (no memory flags, Table 7 hyperparameters)
- Extension run: **same flags + `--natural_gradient`** — single-flag toggle keeps the comparison clean
- With `--natural_gradient` off, the code path is byte-for-byte identical to baseline

### Logging (infrastructure only, no algorithmic effect)

Each training run now writes to `results/{dataset}/{run_name}/`:
- `config.json` — snapshot of all CLI args (once, at training start)
- `metrics.jsonl` — one JSON record per epoch with train metrics, plus test metrics on eval epochs
- `best_model.pth` — existing best-checkpoint behavior, unchanged

`--run_name` (optional): nests outputs under a named subfolder so baseline and extension runs don't clobber each other. Default (unset) preserves the original `results/{dataset}/best_model.pth` path.

In [ ]:
# Cell 7: Smoke test (10 epochs on Tree_Cycle) — fast infrastructure validation +
# wall-time probe to inform epoch budget. Validates:
#   - paper-faithful loop runs end-to-end
#   - Drive symlink redirects writes
#   - config.json + metrics.jsonl + best_model.pth in MyDrive/D4Explainer_results/
#   - per-epoch wall time (printed via "elapsed: Xs" in each Training Epoch line)
# --verbose 5: eval runs at epochs 4 and 9, so 8 epochs show train-only time and 2 show train+eval.

# Baseline smoke
!python main.py --dataset Tree_Cycle --epoch 10 --verbose 5 \
    --n_hidden 64 --num_layers 6 --train_batchsize 32 --alpha_cf 0.1 \
    --run_name smoke_baseline

# Extension smoke (Fisher-Rao natural gradient)
!python main.py --dataset Tree_Cycle --epoch 10 --verbose 5 \
    --n_hidden 64 --num_layers 6 --train_batchsize 32 --alpha_cf 0.1 \
    --natural_gradient --run_name smoke_natgrad


## LR-hypothesis probe (Tree_Cycle, 100 epochs, 2x2 grid)

Smoke at 10 epochs showed natgrad lags baseline on sparsity (avg modification 1.30 vs 0.58)
and on training loss (0.21 vs 0.18). Same pattern as the 2026-04-20 100-epoch run on Mutag.

**Hypothesis:** the Fisher rescaling factor θ(1-θ) has expectation ~0.25 over a random init,
so natgrad effectively shrinks the LR by ~4x vs vanilla SGD. The paper LR (1e-3) is tuned
for vanilla; natgrad needs ~5e-3 to produce a comparable expected step size.

**Design — 2x2 grid (4 arms x 100 epochs, ~33 min):**

| | LR=1e-3 (paper) | LR=5e-3 (compensated) |
|---|---|---|
| **baseline** | `probe_baseline` | `probe_baseline_lr5e3` |
| **natgrad** | `probe_natgrad` | `probe_natgrad_lr5e3` |

Why 2x2 instead of 3 arms: the `baseline_lr5e3` cell confirms the paper LR really is best
for vanilla (i.e., we are not unfairly handicapping baseline by using 1e-3 while comparing
against a tuned-up natgrad). Closes the "you only tuned one method" objection — appendix
will report this 2x2 alongside the headline result. The full pair runs below stay at 2 arms
(baseline at paper LR, natgrad at the LR the probe selects).


In [ ]:
# Cell 7b: Tree_Cycle LR-hypothesis probe — 2x2 grid (baseline/natgrad x 1e-3/5e-3) at 100 epochs.
# Decides which natgrad LR to use in the full pair runs below, and confirms 1e-3 is best
# for vanilla baseline (so we're not handicapping it).

# Arm 1: baseline at paper LR (vanilla SGD baseline, what the paper tuned)
!python main.py --dataset Tree_Cycle --epoch 100 --verbose 10 \
    --n_hidden 64 --num_layers 6 --train_batchsize 32 --alpha_cf 0.1 \
    --run_name probe_baseline 2>&1 | tee results/Tree_Cycle_probe_baseline.log

# Arm 2: baseline at LR=5e-3 (sanity check that 1e-3 really is best for vanilla)
!python main.py --dataset Tree_Cycle --epoch 100 --verbose 10 \
    --n_hidden 64 --num_layers 6 --train_batchsize 32 --alpha_cf 0.1 \
    --learning_rate 0.005 \
    --run_name probe_baseline_lr5e3 2>&1 | tee results/Tree_Cycle_probe_baseline_lr5e3.log

# Arm 3: natgrad at paper LR (replicates smoke lag pattern at 100 epochs)
!python main.py --dataset Tree_Cycle --epoch 100 --verbose 10 \
    --n_hidden 64 --num_layers 6 --train_batchsize 32 --alpha_cf 0.1 \
    --natural_gradient \
    --run_name probe_natgrad 2>&1 | tee results/Tree_Cycle_probe_natgrad.log

# Arm 4: natgrad with LR compensated for the avg Fisher rescaling factor (E[θ(1-θ)] ~ 0.25)
!python main.py --dataset Tree_Cycle --epoch 100 --verbose 10 \
    --n_hidden 64 --num_layers 6 --train_batchsize 32 --alpha_cf 0.1 \
    --natural_gradient --learning_rate 0.005 \
    --run_name probe_natgrad_lr5e3 2>&1 | tee results/Tree_Cycle_probe_natgrad_lr5e3.log

# Read the four metrics.jsonl files and compare final-epoch loss + sparsity to decide.


## Scope-hypothesis probe (Tree_Cycle, 100 epochs, 1 arm)

The 2x2 LR probe falsified the LR-mismatch hypothesis: natgrad stayed pinned at chance
(test CF-acc 0.606) at both LR=1e-3 and LR=5e-3, while baseline at LR=5e-3 was *worse*
than baseline at LR=1e-3. So 5e-3 is not a tuning fix and the natgrad failure is not LR-driven.

**Next hypothesis (CONTEXT.md §2.1):** the Fisher-Rao framing is about the *explanation*
manifold (CF loss domain), not the BCE *reconstruction* objective. Applying θ(1-θ) rescaling
to the per-sigma denoising gradient may be killing the BCE signal that the CF loss depends on.

**Test:** restrict the FR hook to the CF pathway only via `--nat_grad_cf_only`.
BCE flows through the original score un-rescaled; CF flows through hooked clones.
Both contributions sum at the per-sigma score gradient and propagate through the model normally.

**Decision rule:**
- Test CF-acc clearly above 0.606 by epoch 99 → scope hypothesis validated; use `--nat_grad_cf_only`
  in pair runs below.
- Still pinned at 0.606 → reframe the empirical section as a negative result + future-work direction
  (non-diagonal FIM / K-FAC-style approximation; see Discussion).

In [ ]:
# Cell 7c: Tree_Cycle scope-hypothesis probe — natgrad with FR hook restricted to CF pathway only.
# Single arm at paper LR (1e-3). The 2x2 above already established that LR=5e-3 doesn't help
# either method, so re-running both LRs here would just burn time.

!python main.py --dataset Tree_Cycle --epoch 100 --verbose 10 \
    --n_hidden 64 --num_layers 6 --train_batchsize 32 --alpha_cf 0.1 \
    --natural_gradient --nat_grad_cf_only \
    --run_name probe_natgrad_cf_only 2>&1 | tee results/Tree_Cycle_probe_natgrad_cf_only.log

# Compare final-epoch test_acc against:
#   probe_baseline (full hook off): 0.822
#   probe_natgrad (full hook on, LR=1e-3): 0.606  <- chance
# If probe_natgrad_cf_only > 0.65, the scope hypothesis is validated.

## Corrected-hook probe (Tree_Cycle, 100 epochs, 1 arm)

**Bug-fix.** The first FR hook implementation had two issues, both now corrected in
`explainers/natural_gradient.py`:

1. **Logit-vs-probability bug.** `score_batch` is a logit (used by `BCEWithLogitsLoss`,
   sigmoid'd inside `sparsity()`), but the hook clamped it to `[ε, 1-ε]` as if it were a
   probability. Almost all logits got clamped to `ε`, killing the rescaling factor by ~6
   orders of magnitude. The new code applies `sigmoid(logit)` to recover θ first.
2. **Direction bug.** Natural gradient on the θ-parameterised Bernoulli manifold is
   `θ(1-θ) · ∂L/∂θ`. Standard backprop already gives `∂L/∂ℓ = θ(1-θ) · ∂L/∂θ`
   (chain rule through sigmoid). To realise the θ-space natural-gradient direction via
   logit-space SGD, the hook must multiply `∂L/∂ℓ` by `1/(θ(1-θ))`, not `θ(1-θ)`. The
   old hook did the opposite — suppressing exactly the gradient signal that natural
   gradient is supposed to amplify on decided edges. The new hook divides instead of
   multiplies, capped by `--nat_grad_cap` (default 100) to bound update size at extreme
   logits where `1/(θ(1-θ))` would otherwise diverge.

**Decision rule:**
- Test CF-acc converges near baseline (≥ 0.75) → bug-fix is validated; this becomes the
  headline natgrad arm in the pair runs below.
- Still pinned at chance or unstable → escalate to longer probe with different cap, or
  fall back to documented negative result.

In [ ]:
# Cell 7d: Tree_Cycle corrected-hook probe — full natgrad with bug-fixed FR rescaling.
# Default LR (1e-3), default cap (100). Single arm; 2x2 LR sweep can be re-run later if
# this works and we want to show LR insensitivity.

!python main.py --dataset Tree_Cycle --epoch 100 --verbose 10 \
    --n_hidden 64 --num_layers 6 --train_batchsize 32 --alpha_cf 0.1 \
    --natural_gradient \
    --run_name probe_natgrad_corrected 2>&1 | tee results/Tree_Cycle_probe_natgrad_corrected.log

# Compare final-epoch test_acc against:
#   probe_baseline:               0.822 (target)
#   probe_natgrad (old, broken):  0.606 (chance)
#   probe_natgrad_cf_only (old):  bouncing 0.30-0.69, no convergence
# If probe_natgrad_corrected reaches ≥ 0.75 with smooth convergence, the bug-fix story
# is the paper's headline result.

## Paired baseline + natural-gradient runs

One cell per dataset, ordered cheapest to most expensive. Each cell runs the paper-faithful baseline and then the `--natural_gradient` extension with identical hyperparameters. Outputs land in `results/{dataset}/baseline/` and `results/{dataset}/natgrad/`; compare the two `metrics.jsonl` files once both finish.

Run cells top-to-bottom and stop whenever your compute budget runs out. `--epoch` defaults to 800 (paper uses 1500 for Mutagenicity — bump if you want closer-to-paper numbers).

In [ ]:
# Tree_Cycle — Node-class, small graph.
# Paper Table 7: n_hidden 64  num_layers 6 train_batchsize 32 alpha_cf 0.1
# Baseline
!python main.py --epoch 500 --dataset Tree_Cycle --n_hidden 64 --num_layers 6 --train_batchsize 32 --alpha_cf 0.1 --run_name baseline 2>&1 | tee results/Tree_Cycle_baseline.log

# Natural-gradient extension (single-flag toggle: --natural_gradient)
!python main.py --epoch 500 --dataset Tree_Cycle --n_hidden 64 --num_layers 6 --train_batchsize 32 --alpha_cf 0.1 --natural_gradient --run_name natgrad 2>&1 | tee results/Tree_Cycle_natgrad.log


In [ ]:
# Tree_Grids — Node-class, deeper model (num_layers 8).
# Paper Table 7: n_hidden 128 num_layers 8 train_batchsize 32 alpha_cf 0.05
#
# OOM mitigation (2026-05-02): paper config OOMs on A100 due to deeper
# Powerful (num_layers=8) at n_hidden=128. Add --gradient_checkpointing to
# trade compute for memory. Compute-only deviation: same loss, same
# optimizer, same numerical result up to non-determinism. Documented in §4
# setup alongside NCI1 and BA_shapes deviations.
# Expected wall-clock: ~1.5–2x slower per epoch vs unchecked.

# Sanity probe (5 ep) — verify OOM fix works and measure s/ep before launching the pair.
!python main.py --epoch 5 --dataset Tree_Grids --n_hidden 128 --num_layers 8 --train_batchsize 32 --alpha_cf 0.05 --gradient_checkpointing --run_name probe_tree_grids 2>&1 | tee results/Tree_Grids_probe.log

# If probe fits, run the pair (comment out probe above, uncomment below):
# Baseline
# !python main.py --epoch 500 --dataset Tree_Grids --n_hidden 128 --num_layers 8 --train_batchsize 32 --alpha_cf 0.05 --gradient_checkpointing --run_name baseline 2>&1 | tee results/Tree_Grids_baseline.log
# Natural-gradient extension (single-flag toggle: --natural_gradient)
# !python main.py --epoch 500 --dataset Tree_Grids --n_hidden 128 --num_layers 8 --train_batchsize 32 --alpha_cf 0.05 --gradient_checkpointing --natural_gradient --run_name natgrad 2>&1 | tee results/Tree_Grids_natgrad.log


In [ ]:
# cornell — SKIPPED for current run plan (base GNN test acc only ~54%, well below paper Table 6).
# Paper Table 7: n_hidden 128 num_layers 6 train_batchsize 4  alpha_cf 0.05
# Baseline
# !python main.py --dataset cornell --n_hidden 128 --num_layers 6 --train_batchsize 4 --alpha_cf 0.05 --run_name baseline 2>&1 | tee results/cornell_baseline.log

# Natural-gradient extension (single-flag toggle: --natural_gradient)
# !python main.py --dataset cornell --n_hidden 128 --num_layers 6 --train_batchsize 4 --alpha_cf 0.05 --natural_gradient --run_name natgrad 2>&1 | tee results/cornell_natgrad.log


In [ ]:
# NCI1 — Graph-class, ~4k small molecules.
# Paper Table 7: n_hidden 128 num_layers 6 train_batchsize 32 alpha_cf 0.01
#
# OOM on A100 80GB (2026-05-02): even the larger-memory A100 OOMs at paper
# bsz=32. Probable cause is Colab shared-tenant memory overhead reducing
# usable VRAM below the spec. Halve to bsz=16 and add --size_bucketed (tight
# per-batch padding). --test_batchsize 16 matches train so eval also fits.
# Compute-only deviation: same loss, same optimizer, same per-step gradient
# direction; only batch size and bucketing change. Documented in §4 setup
# alongside BA_shapes. If bsz=16 still OOMs, drop to --train_batchsize 8 --test_batchsize 8.

# Sanity probe (5 ep) — verify OOM fix works and measure s/ep before launching the pair.
!python main.py --epoch 5 --dataset NCI1 --n_hidden 128 --num_layers 6 --train_batchsize 16 --test_batchsize 16 --size_bucketed --alpha_cf 0.01 --run_name probe_nci1 2>&1 | tee results/NCI1_probe.log

# If probe fits, run the pair (comment out probe above, uncomment below):
# Baseline
# !python main.py --epoch 500 --dataset NCI1 --n_hidden 128 --num_layers 6 --train_batchsize 16 --test_batchsize 16 --size_bucketed --alpha_cf 0.01 --run_name baseline 2>&1 | tee results/NCI1_baseline.log
# Natural-gradient extension (single-flag toggle: --natural_gradient)
# !python main.py --epoch 500 --dataset NCI1 --n_hidden 128 --num_layers 6 --train_batchsize 16 --test_batchsize 16 --size_bucketed --alpha_cf 0.01 --natural_gradient --run_name natgrad 2>&1 | tee results/NCI1_natgrad.log


In [ ]:
# bbbp — SKIPPED for current run plan (base GNN val acc at chance; weak base => weak explainer comparison).
# Paper Table 7: n_hidden 128 num_layers 6 train_batchsize 16 alpha_cf 0.005
# Baseline
# !python main.py --dataset bbbp --n_hidden 128 --num_layers 6 --train_batchsize 16 --alpha_cf 0.005 --run_name baseline 2>&1 | tee results/bbbp_baseline.log

# Natural-gradient extension (single-flag toggle: --natural_gradient)
# !python main.py --dataset bbbp --n_hidden 128 --num_layers 6 --train_batchsize 16 --alpha_cf 0.005 --natural_gradient --run_name natgrad 2>&1 | tee results/bbbp_natgrad.log


In [ ]:
# ba3 — Graph-class, motif ground truth.
# Paper Table 7: n_hidden 128 num_layers 6 train_batchsize 32 alpha_cf 0.05
# Baseline
!python main.py --epoch 500 --dataset ba3 --n_hidden 128 --num_layers 6 --train_batchsize 32 --alpha_cf 0.05 --run_name baseline 2>&1 | tee results/ba3_baseline.log

# Natural-gradient extension (single-flag toggle: --natural_gradient)
!python main.py --epoch 500 --dataset ba3 --n_hidden 128 --num_layers 6 --train_batchsize 32 --alpha_cf 0.05 --natural_gradient --run_name natgrad 2>&1 | tee results/ba3_natgrad.log


In [ ]:
# BA_shapes — Node-class. COMPUTE-HEAVY: run after the smaller datasets finish.
# Empirical OOM on A100 at paper bsz=4 even with --max_graph_size 450 (peak ~39 GB during
# first forward pass — autograd-retained intermediates dominate, not just the cat tensor).
#
# Deviations from paper Table 7 (which specifies bsz=4):
#   --train_batchsize 2  (halves effective batch sigma_length × bsz from 40 to 20;
#                         predicted peak ~15 GB, fits A100 with margin)
#   --test_batchsize 2   (matches train so eval forward also fits)
#   --max_graph_size 450 (drops upper ~30% from all splits; further safety margin)
# All deviations are compute/data, not loss/optimizer changes.
# Paper Table 7: n_hidden 64  num_layers 6 train_batchsize 4  alpha_cf 0.005
# Baseline
!python main.py --epoch 300 --dataset BA_shapes --n_hidden 64 --num_layers 6 --train_batchsize 2 --test_batchsize 2 --alpha_cf 0.005 --max_graph_size 450 --run_name baseline 2>&1 | tee results/BA_shapes_baseline.log

# Natural-gradient extension (single-flag toggle: --natural_gradient)
!python main.py --epoch 300 --dataset BA_shapes --n_hidden 64 --num_layers 6 --train_batchsize 2 --test_batchsize 2 --alpha_cf 0.005 --max_graph_size 450 --natural_gradient --run_name natgrad 2>&1 | tee results/BA_shapes_natgrad.log


In [ ]:
# mutag — Mutagenicity, SLOWEST: paper bsz=2, max N~417. Falls back to --memory_efficient if OOM.
# Paper Table 7: n_hidden 64  num_layers 6 train_batchsize 2  alpha_cf 0.001
# Baseline
!python main.py --dataset mutag --n_hidden 64 --num_layers 6 --train_batchsize 2 --alpha_cf 0.001 --run_name baseline 2>&1 | tee results/mutag_baseline.log

# Natural-gradient extension (single-flag toggle: --natural_gradient)
!python main.py --dataset mutag --n_hidden 64 --num_layers 6 --train_batchsize 2 --alpha_cf 0.001 --natural_gradient --run_name natgrad 2>&1 | tee results/mutag_natgrad.log

# If Mutag OOMs on your GPU (only expected on T4), re-run with memory-efficient fallback:
# !python main.py --dataset mutag --n_hidden 64 --num_layers 6 --train_batchsize 2 --alpha_cf 0.001 --memory_efficient --size_bucketed --max_graph_size 100 --run_name baseline_me 2>&1 | tee results/mutag_baseline_me.log
# !python main.py --dataset mutag --n_hidden 64 --num_layers 6 --train_batchsize 2 --alpha_cf 0.001 --memory_efficient --size_bucketed --max_graph_size 100 --natural_gradient --run_name natgrad_me 2>&1 | tee results/mutag_natgrad_me.log
